In [ ]:
import pandas as pd
import calendar
import re

indent_file = r"D:/PPC Plan/Monthly Indent/Monthly Indent.xlsx"
bom_file    = r"D:/Tushar/main_with_subs_only.xlsx"

# ───────────────────────────────────────────────
# Read files
# ───────────────────────────────────────────────
try:
    indent_df = pd.read_excel(indent_file)
    bom_df    = pd.read_excel(bom_file)
except Exception as e:
    print("File read error:", e)
    input("Press Enter to exit...")
    exit()

# ─── Fix: Force all column names to be strings + handle NaN headers ───
indent_df.columns = [
    str(c).strip() if pd.notna(c) else f"Unnamed_col_{i}"
    for i, c in enumerate(indent_df.columns)
]
bom_df.columns = [
    str(c).strip() if pd.notna(c) else f"Unnamed_col_{i}"
    for i, c in enumerate(bom_df.columns)
]

# ───────────────────────────────────────────────
# Safe string conversion function
# ───────────────────────────────────────────────
def safe_to_str(series):
    return series.astype(str).replace(['nan', 'NaN', 'None'], '').str.strip()

# ───────────────────────────────────────────────
# Auto-detect possible columns (now safe against non-string columns)
# ───────────────────────────────────────────────
possible_part_cols = [
    c for c in indent_df.columns
    if isinstance(c, str) and any(
        k in c.lower() for k in ['part', 'code', 'fg', 'model', 'item', 'number', 'no.', 'switch']
    )
]

possible_qty_cols = [
    c for c in indent_df.columns
    if isinstance(c, str) and any(
        k in c.lower() for k in ['qty', 'indent', 'plan', 'demand', 'req', 'quantity', 'month']
    )
]

print("Possible part/switch columns in indent file:", possible_part_cols)
print("Possible qty/month columns in indent file :", possible_qty_cols)

# ───────────────────────────────────────────────
# Month column detection (safer version)
# ───────────────────────────────────────────────
pattern = re.compile(r"([A-Za-z]{3})'(\d{2})", re.I)
month_cols = [
    c for c in indent_df.columns
    if isinstance(c, str) and pattern.search(c)
]

if not month_cols:
    print("No month columns like 'Jan'25' found.")
    print("All columns in indent file:", list(indent_df.columns))
    input("Press Enter to exit...")
    exit()

latest_col = max(month_cols, key=str)  # usually the last one is the most recent
print("\nDetected latest month column →", latest_col)

match = pattern.search(latest_col)
if not match:
    print("Cannot parse month/year from:", latest_col)
    exit()

month_str = match.group(1).title()
year = 2000 + int(match.group(2))
month_num = list(calendar.month_abbr).index(month_str)
days_in_month = calendar.monthrange(year, month_num)[1]
print(f"→ {month_str} {year} has {days_in_month} days\n")

# ───────────────────────────────────────────────
# Prepare indent data
# ───────────────────────────────────────────────
part_col = 'Part number'

# Auto fallback if default column not found
if part_col not in indent_df.columns and possible_part_cols:
    part_col = possible_part_cols[0]
    print(f"→ Using detected switch/part column: {part_col}\n")

if part_col not in indent_df.columns:
    print(f"Column '{part_col}' not found.")
    print("Available columns:", list(indent_df.columns))
    input("Press Enter to exit...")
    exit()

# Clean switch codes
indent_df[part_col] = safe_to_str(indent_df[part_col])

indent_df = indent_df[[part_col, latest_col]].dropna(subset=[latest_col])
indent_df[latest_col] = pd.to_numeric(indent_df[latest_col], errors='coerce')
indent_df = indent_df.dropna(subset=[latest_col])

indent_df = indent_df.rename(columns={part_col: 'Switch', latest_col: 'Monthly_Qty'})
indent_df['Daily_Qty'] = indent_df['Monthly_Qty'] / days_in_month

print("Indent data preview (first 8 rows):")
print(indent_df.head(8))
print(f"→ {len(indent_df)} switch types with monthly plan\n")

# ───────────────────────────────────────────────
# Prepare BOM
# ───────────────────────────────────────────────
bom_df = bom_df[['Main_Label', 'Sub_Label', 'Sub_Count']].copy()
bom_df = bom_df.rename(columns={
    'Main_Label': 'Child',
    'Sub_Label': 'Switch',
    'Sub_Count': 'Usage_Qty'
})

bom_df['Switch'] = safe_to_str(bom_df['Switch'])
bom_df['Child']  = safe_to_str(bom_df['Child'])

print(f"BOM has {len(bom_df):,} rows\n")

# ───────────────────────────────────────────────
# Merge BOM with daily demand
# ───────────────────────────────────────────────
merged = pd.merge(
    bom_df,
    indent_df[['Switch', 'Daily_Qty']],
    on='Switch',
    how='inner'
)

print(f"After merge: {len(merged):,} matching rows")

if len(merged) == 0:
    print("\nSample BOM switch codes (first 10):")
    print(sorted(bom_df['Switch'].unique())[:10])
    print("\nSample Indent switch codes (first 10):")
    print(sorted(indent_df['Switch'].unique())[:10])
    print("\n→ No matches found → check formatting (spaces, case, prefixes, zeros, etc.)")
    input("Press Enter to exit...")
    exit()

# Calculate daily child requirement
merged['Daily_Child_Need'] = merged['Daily_Qty'] * merged['Usage_Qty']

# Aggregate per child part
result = merged.groupby('Child', as_index=False)['Daily_Child_Need'].sum()
result = result.rename(columns={'Daily_Child_Need': 'Daily_Qty'})

result['Two_Day_Qty'] = result['Daily_Qty'] * 2

# Formatting & sorting
result = result.sort_values('Two_Day_Qty', ascending=False)
result[['Daily_Qty', 'Two_Day_Qty']] = result[['Daily_Qty', 'Two_Day_Qty']].round(2)

print("\nTop 15 child parts by 2-day requirement:")
print(result.head(15))

# ───────────────────────────────────────────────
# Save result
# ───────────────────────────────────────────────
output_file = "Two_Day_Child_Qty.xlsx"
result.to_excel(output_file, index=False)
print(f"\nSaved to: {output_file}")
print("Done.")